In [0]:
-- LA PHASE SYLVER_LAYER

-- Databricks notebook source
SELECT * FROM bronze_db.raw_trips LIMIT 10;


VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,ingestion_timestamp,file_name
1,2024-03-01T00:18:51Z,2024-03-01T00:23:45Z,0,1.3,1,N,142,239,1,8.6,3.5,0.5,2.7,0.0,1.0,16.3,2.5,0.0,2025-11-13T10:24:42.105334Z,yellow_tripdata_2024_03.parquet
1,2024-03-01T00:26:00Z,2024-03-01T00:29:06Z,0,1.1,1,N,238,24,1,7.2,3.5,0.5,3.0,0.0,1.0,15.2,2.5,0.0,2025-11-13T10:24:42.105334Z,yellow_tripdata_2024_03.parquet
2,2024-03-01T00:09:22Z,2024-03-01T00:15:24Z,1,0.86,1,N,263,75,2,7.9,1.0,0.5,0.0,0.0,1.0,10.4,0.0,0.0,2025-11-13T10:24:42.105334Z,yellow_tripdata_2024_03.parquet
2,2024-03-01T00:33:45Z,2024-03-01T00:39:34Z,1,0.82,1,N,164,162,1,7.9,1.0,0.5,1.29,0.0,1.0,14.19,2.5,0.0,2025-11-13T10:24:42.105334Z,yellow_tripdata_2024_03.parquet
1,2024-03-01T00:05:43Z,2024-03-01T00:26:22Z,0,4.9,1,N,263,7,2,25.4,3.5,0.5,0.0,0.0,1.0,30.4,2.5,0.0,2025-11-13T10:24:42.105334Z,yellow_tripdata_2024_03.parquet
2,2024-03-01T00:50:42Z,2024-03-01T01:10:40Z,1,5.04,1,N,238,159,2,25.4,1.0,0.5,0.0,0.0,1.0,27.9,0.0,0.0,2025-11-13T10:24:42.105334Z,yellow_tripdata_2024_03.parquet
2,2024-03-01T00:08:23Z,2024-03-01T00:17:53Z,1,2.15,1,N,161,141,1,12.1,1.0,0.5,5.13,0.0,1.0,22.23,2.5,0.0,2025-11-13T10:24:42.105334Z,yellow_tripdata_2024_03.parquet
2,2024-03-01T00:24:58Z,2024-03-01T00:30:31Z,1,1.1,1,N,236,237,1,8.6,1.0,0.5,2.04,0.0,1.0,15.64,2.5,0.0,2025-11-13T10:24:42.105334Z,yellow_tripdata_2024_03.parquet
2,2024-03-01T00:49:40Z,2024-03-01T01:01:25Z,1,2.78,1,N,161,114,1,14.9,1.0,0.5,2.0,0.0,1.0,21.9,2.5,0.0,2025-11-13T10:24:42.105334Z,yellow_tripdata_2024_03.parquet
1,2024-03-01T00:21:43Z,2024-03-01T00:24:44Z,1,0.3,1,N,237,141,2,5.1,3.5,0.5,0.0,0.0,1.0,10.1,2.5,0.0,2025-11-13T10:24:42.105334Z,yellow_tripdata_2024_03.parquet


In [0]:
CREATE DATABASE IF NOT EXISTS silver_db;

show databases;

databaseName
bronze_db
default
information_schema
silver_db
tripsdata


In [0]:
USE silver_db;

SELECT current_database();

current_schema()
silver_db


In [0]:
-- La regle à appliquer pour la couche silver_layer
-- passenger_count > 0
-- trip_distance > 0
-- total_amount > 0
-- Date et heure de demarrage doit être antérieure à date et heure de fin du trajet 
-- considerer uniquement les paiements en carte de crédit

## Pipeline Silver incrémental

## 1. Préparer la table Silver (si elle n'existe pas encore)

In [0]:
CREATE TABLE IF NOT EXISTS processed_trips 
USING DELTA
AS
SELECT 
  *
FROM bronze_db.raw_trips
WHERE 
  passenger_count > 0
  AND trip_distance > 0
  AND total_amount > 0
  AND tpep_pickup_datetime < tpep_dropoff_datetime 
  AND tip_amount >= 0
  AND payment_type = 1;


num_affected_rows,num_inserted_rows


In [0]:
SELECT COUNT(*) FROM processed_trips

count(1)
9617774


In [0]:
-- Afficher un extrait de la table
select * from processed_trips LIMIT 10;

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,ingestion_timestamp,file_name
2,2024-04-01T00:41:12Z,2024-04-01T00:55:29Z,1,5.6,1,N,264,264,1,25.4,1.0,0.5,10.0,0.0,1.0,37.9,0.0,0.0,2025-11-13T11:44:25.184563Z,yellow_tripdata_2024_04.parquet
2,2024-04-01T00:48:42Z,2024-04-01T01:05:30Z,1,3.55,1,N,186,236,1,20.5,1.0,0.5,5.1,0.0,1.0,30.6,2.5,0.0,2025-11-13T11:44:25.184563Z,yellow_tripdata_2024_04.parquet
1,2024-04-01T00:08:32Z,2024-04-01T00:10:24Z,1,0.7,1,N,236,263,1,5.1,3.5,0.5,2.0,0.0,1.0,12.1,2.5,0.0,2025-11-13T11:44:25.184563Z,yellow_tripdata_2024_04.parquet
1,2024-04-01T00:02:09Z,2024-04-01T00:20:33Z,1,4.7,1,N,138,146,1,21.9,7.75,0.5,7.8,0.0,1.0,38.95,0.0,1.75,2025-11-13T11:44:25.184563Z,yellow_tripdata_2024_04.parquet
2,2024-04-01T00:27:16Z,2024-04-01T00:43:44Z,2,3.24,1,N,186,87,1,18.4,1.0,0.5,4.68,0.0,1.0,28.08,2.5,0.0,2025-11-13T11:44:25.184563Z,yellow_tripdata_2024_04.parquet
1,2024-04-01T00:41:26Z,2024-04-01T01:25:31Z,1,21.5,2,N,132,143,1,70.0,4.25,0.5,16.54,6.94,1.0,99.23,2.5,1.75,2025-11-13T11:44:25.184563Z,yellow_tripdata_2024_04.parquet
2,2024-04-01T00:19:04Z,2024-04-01T00:29:25Z,3,2.39,1,N,249,230,1,13.5,1.0,0.5,3.7,0.0,1.0,22.2,2.5,0.0,2025-11-13T11:44:25.184563Z,yellow_tripdata_2024_04.parquet
2,2024-04-01T00:35:37Z,2024-04-01T00:41:16Z,4,1.43,1,N,100,113,1,8.6,1.0,0.5,2.04,0.0,1.0,15.64,2.5,0.0,2025-11-13T11:44:25.184563Z,yellow_tripdata_2024_04.parquet
2,2024-04-01T00:23:35Z,2024-04-01T00:34:58Z,1,2.65,1,N,161,239,1,14.2,1.0,0.5,3.84,0.0,1.0,23.04,2.5,0.0,2025-11-13T11:44:25.184563Z,yellow_tripdata_2024_04.parquet
2,2024-04-01T00:33:47Z,2024-04-01T00:38:40Z,1,1.51,1,N,137,141,1,8.6,1.0,0.5,2.72,0.0,1.0,16.32,2.5,0.0,2025-11-13T11:44:25.184563Z,yellow_tripdata_2024_04.parquet


In [0]:
-- Verification si les régles sont appliquées
SELECT * FROM processed_trips WHERE trip_distance < 0;

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,ingestion_timestamp,file_name


In [0]:
-- Afficher les fichiers disponibles dans la table processed_trips
SELECT DISTINCT file_name FROM processed_trips ORDER BY file_name;

file_name
yellow_tripdata_2024_01.parquet
yellow_tripdata_2024_02.parquet
yellow_tripdata_2024_03.parquet
yellow_tripdata_2024_04.parquet


## 2. Charger seulement les nouvelles données de la table Bronze

In [0]:
-- Afficher le maximum de la colonne "ingestion_timestamp"
SELECT max(ingestion_timestamp) FROM processed_trips;

max(ingestion_timestamp)
2025-11-13T11:44:25.184563Z


In [0]:

WITH new_data AS (
  SELECT 
    * 
  FROM bronze_db.raw_trips b  
  WHERE b.ingestion_timestamp > (
    SELECT MAX(ingestion_timestamp) 
    FROM processed_trips
  )
  AND b.passenger_count > 0
  AND b.trip_distance > 0
  AND b.total_amount > 0
  AND b.tpep_pickup_datetime < tpep_dropoff_datetime 
  AND b.tip_amount >= 0
  AND b.payment_type = 1
)

MERGE INTO processed_trips s
USING new_data n
ON s.file_name = n.file_name 
AND s.tpep_pickup_datetime = n.tpep_pickup_datetime
AND s.tpep_dropoff_datetime = n.tpep_dropoff_datetime
AND s.ingestion_timestamp = n.ingestion_timestamp
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2674590,0,0,2674590


In [0]:
SELECT COUNT(*) FROM processed_trips;

count(1)
12292364


In [0]:
SELECT DISTINCT file_name FROM processed_trips ORDER BY file_name;

file_name
yellow_tripdata_2024_01.parquet
yellow_tripdata_2024_02.parquet
yellow_tripdata_2024_03.parquet
yellow_tripdata_2024_04.parquet
yellow_tripdata_2024_05.parquet
